# Advanced Flight Delay Analysis: Geospatial Bottlenecks & Holiday Anomalies

## Focused Deep-Dive Analysis

**Focus Areas**:
1. **Geospatial Analysis**: Major hub bottlenecks (ATL, ORD, LAX) and critical routes
2. **Temporal Anomalies**: Holiday disruptions and event-driven delay patterns

**Visualization Approach**: Complex, multi-dimensional, interactive visualizations using advanced graphing libraries.

---

## Setup & Data Loading

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Network and geospatial libraries
import networkx as nx
from scipy.stats import zscore

# Configure
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)

print("✓ Libraries loaded")

In [ ]:
# Define paths
PROJECT_ROOT = Path('/home/user/Flight-delay')
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Load data
df = pd.read_parquet(DATA_PROCESSED / 'flights_active.parquet')

print(f"✓ Loaded {len(df):,} flights")
print(f"  Columns: {len(df.columns)}")
print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Create date column if not exists
date_cols = [col for col in df.columns if 'DATE' in col.upper() or 'FL_DATE' in col.upper()]

if date_cols:
    date_col = date_cols[0]
    df[date_col] = pd.to_datetime(df[date_col])
    print(f"Using date column: {date_col}")
elif all(col in df.columns for col in ['YEAR', 'MONTH', 'DAY_OF_MONTH']):
    df['FL_DATE'] = pd.to_datetime(df[['YEAR', 'MONTH', 'DAY_OF_MONTH']].rename(
        columns={'YEAR': 'year', 'MONTH': 'month', 'DAY_OF_MONTH': 'day'}))
    date_col = 'FL_DATE'
    print(f"✓ Created FL_DATE column")
else:
    print("⚠️  Warning: No date column available")
    date_col = None

In [ ]:
# US Airport Coordinates (major airports)
# This dictionary contains lat/lon for visualization purposes
AIRPORT_COORDS = {
    'ATL': (33.6407, -84.4277),   # Atlanta
    'ORD': (41.9742, -87.9073),   # Chicago O'Hare
    'LAX': (33.9416, -118.4085),  # Los Angeles
    'DFW': (32.8998, -97.0403),   # Dallas/Fort Worth
    'DEN': (39.8561, -104.6737),  # Denver
    'JFK': (40.6413, -73.7781),   # New York JFK
    'SFO': (37.6213, -122.3790),  # San Francisco
    'LAS': (36.0840, -115.1537),  # Las Vegas
    'SEA': (47.4502, -122.3088),  # Seattle
    'MCO': (28.4312, -81.3081),   # Orlando
    'EWR': (40.6895, -74.1745),   # Newark
    'CLT': (35.2144, -80.9473),   # Charlotte
    'PHX': (33.4346, -112.0116),  # Phoenix
    'IAH': (29.9902, -95.3368),   # Houston
    'MIA': (25.7959, -80.2870),   # Miami
    'BOS': (42.3656, -71.0096),   # Boston
    'MSP': (44.8848, -93.2223),   # Minneapolis
    'DTW': (42.2162, -83.3554),   # Detroit
    'PHL': (39.8744, -75.2424),   # Philadelphia
    'LGA': (40.7769, -73.8740),   # LaGuardia
    'BWI': (39.1774, -76.6684),   # Baltimore
    'SLC': (40.7899, -111.9791),  # Salt Lake City
    'DCA': (38.8521, -77.0377),   # Washington Reagan
    'SAN': (32.7338, -117.1933),  # San Diego
    'TPA': (27.9755, -82.5332),   # Tampa
}

print(f"✓ Airport coordinates loaded for {len(AIRPORT_COORDS)} major airports")

In [ ]:
# 2024 US Holiday Calendar
HOLIDAYS_2024 = {
    'New Year': ['2024-01-01'],
    'MLK Day': ['2024-01-15'],
    'Presidents Day': ['2024-02-19'],
    'Spring Break': ['2024-03-10', '2024-03-11', '2024-03-12', '2024-03-13', '2024-03-14', 
                     '2024-03-15', '2024-03-16', '2024-03-17'],  # Peak spring break week
    'Easter': ['2024-03-29', '2024-03-30', '2024-03-31'],
    'Memorial Day': ['2024-05-25', '2024-05-26', '2024-05-27'],
    'Independence Day': ['2024-07-03', '2024-07-04', '2024-07-05'],
    'Labor Day': ['2024-08-31', '2024-09-01', '2024-09-02'],
    'Thanksgiving': ['2024-11-27', '2024-11-28', '2024-11-29', '2024-11-30', '2024-12-01'],
    'Christmas': ['2024-12-23', '2024-12-24', '2024-12-25', '2024-12-26', '2024-12-27']
}

# Flatten to all holiday dates
all_holiday_dates = []
for holiday, dates in HOLIDAYS_2024.items():
    all_holiday_dates.extend([pd.to_datetime(d) for d in dates])

all_holiday_dates = set(all_holiday_dates)

print(f"✓ Holiday calendar defined: {len(HOLIDAYS_2024)} holiday periods")
print(f"  Total holiday dates: {len(all_holiday_dates)}")

In [ ]:
# Helper function to save visualizations
def save_viz(fig, filename, dpi=300):
    """Save visualization to reports directory."""
    filepath = REPORTS_DIR / f"{filename}.png"
    
    if hasattr(fig, 'write_image'):  # Plotly
        fig.write_image(str(filepath), width=1400, height=800)
    else:  # Matplotlib
        fig.savefig(filepath, dpi=dpi, bbox_inches='tight')
    
    print(f"✓ Saved: {filepath.name}")

print("Helper function loaded")

---

# PART 1: GEOSPATIAL BOTTLENECK ANALYSIS

## Question 1: Which airports (especially major hubs like ATL, ORD, LAX) and routes are the most significant bottlenecks in the system?

**Analysis Strategy**:
1. Identify major hub performance and delay characteristics
2. Visualize geographic distribution of delays
3. Analyze route-level bottlenecks with network visualization
4. Compare hub vs non-hub delay patterns
5. Identify critical chokepoints in the network

## Visualization 1A: Interactive Geographic Heatmap of Airport Delays

**Purpose**: Show geographic distribution of delay severity across major US airports.

**Complexity**: 
- Interactive map with bubble size = volume, color = delay severity
- Major hubs (ATL, ORD, LAX, DFW, DEN) highlighted
- Tooltips with detailed statistics

In [ ]:
# Calculate airport-level statistics
airport_stats = df.groupby('ORIGIN').agg({
    'DEP_DELAY': ['mean', 'median', 'std'],
    'ARR_DELAY': 'mean',
    'ORIGIN': 'count'
}).round(2)

airport_stats.columns = ['Mean_Dep_Delay', 'Median_Dep_Delay', 'Std_Dep_Delay', 'Mean_Arr_Delay', 'Flight_Count']
airport_stats = airport_stats.reset_index()

# Calculate delay rate (% flights delayed ≥15 min)
delay_rates = df.groupby('ORIGIN').apply(
    lambda x: (x['DEP_DELAY'] >= 15).sum() / len(x) * 100
).reset_index(name='Delay_Rate')

airport_stats = airport_stats.merge(delay_rates, on='ORIGIN')

# Add coordinates
airport_stats['Lat'] = airport_stats['ORIGIN'].map(lambda x: AIRPORT_COORDS.get(x, (None, None))[0])
airport_stats['Lon'] = airport_stats['ORIGIN'].map(lambda x: AIRPORT_COORDS.get(x, (None, None))[1])

# Filter to airports with coordinates
airport_stats_geo = airport_stats[airport_stats['Lat'].notna()].copy()

# Mark major hubs
major_hubs = ['ATL', 'ORD', 'LAX', 'DFW', 'DEN']
airport_stats_geo['Is_Major_Hub'] = airport_stats_geo['ORIGIN'].isin(major_hubs)
airport_stats_geo['Hub_Label'] = airport_stats_geo.apply(
    lambda x: f"{x['ORIGIN']} (MAJOR HUB)" if x['Is_Major_Hub'] else x['ORIGIN'], axis=1
)

print(f"\n=== AIRPORT DELAY STATISTICS ===")
print(f"Total airports analyzed: {len(airport_stats)}")
print(f"Airports with geo coordinates: {len(airport_stats_geo)}")
print(f"\nTop 10 Airports by Delay Rate:")
print(airport_stats.nlargest(10, 'Delay_Rate')[['ORIGIN', 'Mean_Dep_Delay', 'Delay_Rate', 'Flight_Count']])

In [ ]:
# Create interactive geographic bubble map
fig = px.scatter_geo(
    airport_stats_geo,
    lat='Lat',
    lon='Lon',
    size='Flight_Count',
    color='Mean_Dep_Delay',
    hover_name='ORIGIN',
    hover_data={
        'Mean_Dep_Delay': ':.2f',
        'Median_Dep_Delay': ':.2f',
        'Delay_Rate': ':.1f',
        'Flight_Count': ':,',
        'Lat': False,
        'Lon': False
    },
    size_max=50,
    color_continuous_scale='RdYlGn_r',  # Red = high delay, Green = low delay
    color_continuous_midpoint=0,
    title='<b>US Airport Delay Heatmap: Geographic Distribution of Bottlenecks</b><br>' + 
          '<sub>Bubble Size = Flight Volume | Color = Mean Departure Delay (minutes)</sub>',
    scope='usa',
)

# Highlight major hubs with annotations
for _, row in airport_stats_geo[airport_stats_geo['Is_Major_Hub']].iterrows():
    fig.add_annotation(
        x=row['Lon'],
        y=row['Lat'],
        text=row['ORIGIN'],
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
        arrowcolor='red',
        font=dict(size=12, color='red', family='Arial Black'),
        bgcolor='white',
        opacity=0.8
    )

fig.update_layout(
    height=800,
    geo=dict(
        projection_type='albers usa',
        showland=True,
        landcolor='rgb(243, 243, 243)',
        coastlinecolor='rgb(204, 204, 204)',
    ),
    coloraxis_colorbar=dict(
        title='Mean Delay<br>(minutes)',
        ticksuffix=' min'
    )
)

save_viz(fig, 'geo_01_airport_heatmap')
fig.show()

**Interpretation**:

*Geographic patterns visible in the map reveal:*
- Distribution of delay severity across US regions
- Major hub performance relative to smaller airports
- Regional clusters of high-delay airports
- Relationship between volume and delay severity

## Visualization 1B: Major Hub Deep Dive - Multi-Metric Comparison

**Purpose**: Detailed comparison of major hubs (ATL, ORD, LAX, DFW, DEN) across multiple delay metrics.

**Complexity**: Radar/Spider chart showing 6 performance dimensions

In [ ]:
# Focus on major hubs
hub_data = df[df['ORIGIN'].isin(major_hubs)].copy()

# Calculate comprehensive metrics
hub_metrics = hub_data.groupby('ORIGIN').agg({
    'DEP_DELAY': ['mean', 'median'],
    'ARR_DELAY': 'mean',
    'ORIGIN': 'count'
}).round(2)

hub_metrics.columns = ['Mean_Dep_Delay', 'Median_Dep_Delay', 'Mean_Arr_Delay', 'Flight_Count']
hub_metrics = hub_metrics.reset_index()

# Calculate additional metrics
hub_metrics['Delay_Rate'] = hub_data.groupby('ORIGIN').apply(
    lambda x: (x['DEP_DELAY'] >= 15).sum() / len(x) * 100
).values

hub_metrics['Severe_Delay_Rate'] = hub_data.groupby('ORIGIN').apply(
    lambda x: (x['DEP_DELAY'] >= 60).sum() / len(x) * 100
).values

hub_metrics['On_Time_Rate'] = 100 - hub_metrics['Delay_Rate']

print("=== MAJOR HUB COMPARISON ===")
print(hub_metrics.to_string(index=False))

In [ ]:
# Create radar chart for hub comparison
fig = go.Figure()

# Normalize metrics for radar chart (0-100 scale)
metrics_to_plot = ['On_Time_Rate', 'Delay_Rate', 'Severe_Delay_Rate', 
                   'Mean_Dep_Delay', 'Median_Dep_Delay', 'Mean_Arr_Delay']

# Create normalized version
hub_metrics_norm = hub_metrics.copy()
for metric in ['Mean_Dep_Delay', 'Median_Dep_Delay', 'Mean_Arr_Delay']:
    # Normalize to 0-100 (invert so lower delay = higher score)
    max_val = hub_metrics[metric].max()
    hub_metrics_norm[f'{metric}_Norm'] = 100 - (hub_metrics[metric] / max_val * 100)

# Radar chart categories
categories = ['On-Time<br>Rate (%)', 'Delay<br>Rate (%)', 'Severe Delay<br>Rate (%)',
              'Dep Delay<br>(Inverted)', 'Median Delay<br>(Inverted)', 'Arr Delay<br>(Inverted)']

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for i, airport in enumerate(major_hubs):
    row = hub_metrics_norm[hub_metrics_norm['ORIGIN'] == airport].iloc[0]
    
    values = [
        row['On_Time_Rate'],
        row['Delay_Rate'],
        row['Severe_Delay_Rate'],
        row['Mean_Dep_Delay_Norm'],
        row['Median_Dep_Delay_Norm'],
        row['Mean_Arr_Delay_Norm']
    ]
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories,
        fill='toself',
        name=airport,
        line=dict(color=colors[i], width=2),
        opacity=0.6
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100]
        )
    ),
    showlegend=True,
    title='<b>Major Hub Performance Comparison: Multi-Metric Radar Analysis</b><br>' +
          '<sub>Higher values = Better performance (delays inverted for comparability)</sub>',
    height=700,
    width=900
)

save_viz(fig, 'geo_02_hub_radar')
fig.show()

**Interpretation**:

*Radar chart reveals:*
- Which hub has the best overall performance (largest polygon)
- Strengths and weaknesses of each major hub
- Consistency across metrics (even vs uneven polygon)
- Specific problem areas for each hub

## Visualization 1C: Route Network Bottleneck Analysis

**Purpose**: Visualize the flight network showing most problematic routes.

**Complexity**: 
- Network graph with nodes = airports, edges = routes
- Edge thickness = flight volume
- Edge color = mean delay
- Top 50 most delayed routes highlighted

In [ ]:
# Calculate route-level statistics
route_stats = df.groupby(['ORIGIN', 'DEST']).agg({
    'ARR_DELAY': 'mean',
    'DEP_DELAY': 'mean',
    'ORIGIN': 'count'
}).round(2)

route_stats.columns = ['Mean_Arr_Delay', 'Mean_Dep_Delay', 'Flight_Count']
route_stats = route_stats.reset_index()

# Filter to routes with sufficient volume (≥50 flights) and known coordinates
route_stats = route_stats[route_stats['Flight_Count'] >= 50]
route_stats = route_stats[
    route_stats['ORIGIN'].isin(AIRPORT_COORDS.keys()) & 
    route_stats['DEST'].isin(AIRPORT_COORDS.keys())
]

# Get top 50 most delayed routes
top_delayed_routes = route_stats.nlargest(50, 'Mean_Arr_Delay')

print(f"\n=== ROUTE BOTTLENECK ANALYSIS ===")
print(f"Total routes analyzed: {len(route_stats):,}")
print(f"\nTop 10 Most Delayed Routes:")
print(top_delayed_routes.head(10)[['ORIGIN', 'DEST', 'Mean_Arr_Delay', 'Flight_Count']].to_string(index=False))

In [ ]:
# Create network graph visualization using plotly
fig = go.Figure()

# Add routes as lines (edges)
for _, route in top_delayed_routes.iterrows():
    origin_coords = AIRPORT_COORDS[route['ORIGIN']]
    dest_coords = AIRPORT_COORDS[route['DEST']]
    
    # Determine line color based on delay severity
    delay = route['Mean_Arr_Delay']
    if delay > 30:
        color = 'rgba(231, 76, 60, 0.7)'  # Red - severe
    elif delay > 15:
        color = 'rgba(230, 126, 34, 0.6)'  # Orange - moderate
    else:
        color = 'rgba(241, 196, 15, 0.5)'  # Yellow - mild
    
    # Line width based on volume (normalized)
    width = min(route['Flight_Count'] / 100, 5)
    
    fig.add_trace(go.Scattergeo(
        lon=[origin_coords[1], dest_coords[1]],
        lat=[origin_coords[0], dest_coords[0]],
        mode='lines',
        line=dict(width=width, color=color),
        hoverinfo='text',
        text=f"{route['ORIGIN']}→{route['DEST']}<br>" +
             f"Mean Delay: {delay:.1f} min<br>" +
             f"Flights: {route['Flight_Count']:,}",
        showlegend=False
    ))

# Add airport nodes
airports_in_network = set(top_delayed_routes['ORIGIN'].tolist() + top_delayed_routes['DEST'].tolist())
airport_lats = [AIRPORT_COORDS[a][0] for a in airports_in_network]
airport_lons = [AIRPORT_COORDS[a][1] for a in airports_in_network]
airport_names = list(airports_in_network)

# Size airports by how many problematic routes they're involved in
airport_involvement = {}
for airport in airports_in_network:
    count = len(top_delayed_routes[
        (top_delayed_routes['ORIGIN'] == airport) | 
        (top_delayed_routes['DEST'] == airport)
    ])
    airport_involvement[airport] = count

airport_sizes = [airport_involvement[a] * 3 for a in airport_names]

fig.add_trace(go.Scattergeo(
    lon=airport_lons,
    lat=airport_lats,
    mode='markers+text',
    marker=dict(
        size=airport_sizes,
        color='rgba(52, 152, 219, 0.8)',
        line=dict(width=2, color='white'),
        sizemode='diameter'
    ),
    text=airport_names,
    textfont=dict(size=10, color='black', family='Arial Black'),
    textposition='top center',
    hoverinfo='text',
    hovertext=[f"{a}<br>Problematic routes: {airport_involvement[a]}" for a in airport_names],
    name='Airports'
))

fig.update_layout(
    title='<b>Flight Network Bottleneck Map: Top 50 Most Delayed Routes</b><br>' +
          '<sub>Line Color: Red = Severe Delay, Orange = Moderate, Yellow = Mild | ' +
          'Line Width = Flight Volume | Node Size = # Problematic Routes</sub>',
    showlegend=False,
    geo=dict(
        scope='usa',
        projection_type='albers usa',
        showland=True,
        landcolor='rgb(243, 243, 243)',
        coastlinecolor='rgb(204, 204, 204)',
        bgcolor='rgb(250, 250, 250)'
    ),
    height=800,
    width=1400
)

save_viz(fig, 'geo_03_network_bottlenecks')
fig.show()

**Interpretation**:

*Network visualization reveals:*
- Geographic concentration of problematic routes
- Which hubs are central to bottleneck routes
- Common origin-destination pairs with persistent delays
- Whether delays are route-specific or airport-specific

## Visualization 1D: Hub vs Route-Level Bottleneck Matrix

**Purpose**: Show which hub-to-hub connections are most problematic.

**Complexity**: Heatmap matrix of major hubs showing route delay severity

In [ ]:
# Create hub-to-hub delay matrix
major_hubs_extended = ['ATL', 'ORD', 'LAX', 'DFW', 'DEN', 'JFK', 'SFO', 'SEA', 'PHX', 'CLT']

hub_routes = df[
    df['ORIGIN'].isin(major_hubs_extended) & 
    df['DEST'].isin(major_hubs_extended)
]

# Create pivot table
hub_matrix = hub_routes.pivot_table(
    values='ARR_DELAY',
    index='ORIGIN',
    columns='DEST',
    aggfunc='mean'
)

# Reorder by total delay involvement
hub_order = hub_matrix.mean(axis=1).sort_values(ascending=False).index
hub_matrix = hub_matrix.reindex(index=hub_order, columns=hub_order)

# Create annotated heatmap
fig, ax = plt.subplots(figsize=(14, 12))

# Mask diagonal (self-routes don't exist)
mask = np.zeros_like(hub_matrix, dtype=bool)
np.fill_diagonal(mask, True)

sns.heatmap(
    hub_matrix,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn_r',
    center=0,
    linewidths=0.5,
    linecolor='gray',
    cbar_kws={'label': 'Mean Arrival Delay (minutes)'},
    mask=mask,
    vmin=-10,
    vmax=30,
    ax=ax
)

ax.set_title(
    'Hub-to-Hub Route Delay Matrix: Identifying Critical Bottleneck Corridors\n' +
    'Red = High Delays | Green = On-Time | White = No Direct Route',
    fontsize=14,
    fontweight='bold',
    pad=20
)
ax.set_xlabel('Destination Hub', fontsize=12, fontweight='bold')
ax.set_ylabel('Origin Hub', fontsize=12, fontweight='bold')

plt.tight_layout()
save_viz(fig, 'geo_04_hub_matrix')
plt.show()

print("\n=== WORST HUB-TO-HUB ROUTES ===")
hub_routes_summary = hub_routes.groupby(['ORIGIN', 'DEST']).agg({
    'ARR_DELAY': 'mean',
    'ORIGIN': 'count'
}).round(2)
hub_routes_summary.columns = ['Mean_Delay', 'Flight_Count']
hub_routes_summary = hub_routes_summary.reset_index().nlargest(10, 'Mean_Delay')
print(hub_routes_summary.to_string(index=False))

**Interpretation**:

*Hub-to-hub matrix reveals:*
- Which hub pairs consistently have delays
- Asymmetric patterns (e.g., A→B delayed but B→A on-time)
- Systematic corridors requiring operational attention
- Whether delays are bidirectional or unidirectional

---

# PART 2: HOLIDAY & TEMPORAL ANOMALY ANALYSIS

## Question 2: How do major disruptive events such as holidays distort "normal" delay patterns?

**Analysis Strategy**:
1. Define normal vs holiday periods
2. Compare delay distributions
3. Identify specific holiday effects (Thanksgiving, Christmas, etc.)
4. Analyze pre-holiday, during-holiday, post-holiday patterns
5. Measure magnitude of disruption

## Visualization 2A: Holiday Impact Timeline - Annotated Time Series

**Purpose**: Show how delays fluctuate throughout 2024 with holiday periods highlighted.

**Complexity**: 
- Time series with holiday shading
- Multiple metrics: mean delay, delay rate, severe delay rate
- Statistical bands (confidence intervals)
- Event annotations

In [ ]:
# Prepare daily time series data
if date_col:
    daily_data = df.groupby(date_col).agg({
        'DEP_DELAY': ['mean', 'std'],
        'ARR_DELAY': 'mean',
        'ORIGIN': 'count'
    }).reset_index()
    
    daily_data.columns = ['Date', 'Mean_Dep_Delay', 'Std_Dep_Delay', 'Mean_Arr_Delay', 'Flight_Count']
    
    # Calculate delay rate
    daily_delay_rate = df.groupby(date_col).apply(
        lambda x: (x['DEP_DELAY'] >= 15).sum() / len(x) * 100
    ).reset_index(name='Delay_Rate')
    daily_delay_rate.columns = ['Date', 'Delay_Rate']
    
    daily_data = daily_data.merge(daily_delay_rate, on='Date')
    
    # Mark holiday periods
    daily_data['Is_Holiday'] = daily_data['Date'].isin(all_holiday_dates)
    
    # Calculate 7-day rolling average for smoothing
    daily_data['Mean_Delay_MA7'] = daily_data['Mean_Dep_Delay'].rolling(window=7, center=True).mean()
    daily_data['Delay_Rate_MA7'] = daily_data['Delay_Rate'].rolling(window=7, center=True).mean()
    
    print(f"✓ Daily time series prepared: {len(daily_data)} days")
    print(f"  Holiday days: {daily_data['Is_Holiday'].sum()}")
    print(f"  Normal days: {(~daily_data['Is_Holiday']).sum()}")
else:
    print("⚠️  Cannot create time series without date column")

In [ ]:
# Create comprehensive timeline visualization
if date_col:
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=(
            'Mean Departure Delay Over Time',
            'Delay Rate (% Flights Delayed ≥15 min)'
        ),
        vertical_spacing=0.12,
        row_heights=[0.5, 0.5]
    )
    
    # Plot 1: Mean delay
    # Add shaded regions for holidays
    for holiday, dates in HOLIDAYS_2024.items():
        if dates:
            start_date = pd.to_datetime(dates[0])
            end_date = pd.to_datetime(dates[-1])
            
            fig.add_vrect(
                x0=start_date, x1=end_date,
                fillcolor='rgba(255, 0, 0, 0.1)',
                layer='below',
                line_width=0,
                row=1, col=1
            )
            fig.add_vrect(
                x0=start_date, x1=end_date,
                fillcolor='rgba(255, 0, 0, 0.1)',
                layer='below',
                line_width=0,
                row=2, col=1
            )
    
    # Daily values (light)
    fig.add_trace(
        go.Scatter(
            x=daily_data['Date'],
            y=daily_data['Mean_Dep_Delay'],
            mode='lines',
            name='Daily Mean',
            line=dict(color='lightblue', width=1),
            opacity=0.4
        ),
        row=1, col=1
    )
    
    # 7-day moving average (bold)
    fig.add_trace(
        go.Scatter(
            x=daily_data['Date'],
            y=daily_data['Mean_Delay_MA7'],
            mode='lines',
            name='7-Day Avg',
            line=dict(color='blue', width=3)
        ),
        row=1, col=1
    )
    
    # Plot 2: Delay rate
    fig.add_trace(
        go.Scatter(
            x=daily_data['Date'],
            y=daily_data['Delay_Rate'],
            mode='lines',
            name='Daily Rate',
            line=dict(color='lightcoral', width=1),
            opacity=0.4
        ),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=daily_data['Date'],
            y=daily_data['Delay_Rate_MA7'],
            mode='lines',
            name='7-Day Avg Rate',
            line=dict(color='red', width=3)
        ),
        row=2, col=1
    )
    
    # Add annotations for major holidays
    major_holiday_dates = {
        'Thanksgiving': '2024-11-28',
        'Christmas': '2024-12-25',
        'July 4th': '2024-07-04',
        'Memorial Day': '2024-05-27'
    }
    
    for holiday, date_str in major_holiday_dates.items():
        holiday_date = pd.to_datetime(date_str)
        if holiday_date in daily_data['Date'].values:
            delay_val = daily_data[daily_data['Date'] == holiday_date]['Mean_Delay_MA7'].values[0]
            
            fig.add_annotation(
                x=holiday_date,
                y=delay_val,
                text=holiday,
                showarrow=True,
                arrowhead=2,
                arrowcolor='red',
                font=dict(size=10, color='red'),
                row=1, col=1
            )
    
    # Update layout
    fig.update_xaxes(title_text='Date', row=2, col=1)
    fig.update_yaxes(title_text='Mean Delay (minutes)', row=1, col=1)
    fig.update_yaxes(title_text='Delay Rate (%)', row=2, col=1)
    
    fig.update_layout(
        title='<b>2024 Flight Delay Timeline: Holiday Disruption Analysis</b><br>' +
              '<sub>Red Shaded Regions = Holiday Periods | Red Line = 7-Day Moving Average</sub>',
        height=900,
        width=1400,
        hovermode='x unified',
        showlegend=True
    )
    
    save_viz(fig, 'anomaly_01_holiday_timeline')
    fig.show()
else:
    print("⚠️  Skipping timeline visualization")

**Interpretation**:

*Timeline analysis reveals:*
- Temporal clustering of delays around holiday periods
- Magnitude of holiday disruption vs baseline
- Recovery time after holiday periods
- Which holidays cause the most severe disruptions

## Visualization 2B: Holiday vs Normal Distribution Comparison

**Purpose**: Statistically compare delay distributions between normal and holiday periods.

**Complexity**: 
- Overlapping distributions (KDE + histogram)
- Statistical tests (KS-test, t-test)
- Percentile comparisons
- Multiple holiday period breakdown

In [ ]:
# Prepare holiday vs normal data
if date_col:
    df['Is_Holiday'] = df[date_col].isin(all_holiday_dates)
    
    normal_delays = df[~df['Is_Holiday']]['DEP_DELAY'].dropna()
    holiday_delays = df[df['Is_Holiday']]['DEP_DELAY'].dropna()
    
    # Filter to reasonable range for visualization
    normal_delays_plot = normal_delays[normal_delays.between(-30, 120)]
    holiday_delays_plot = holiday_delays[holiday_delays.between(-30, 120)]
    
    print("\n=== HOLIDAY VS NORMAL COMPARISON ===")
    print(f"\nNormal Days:")
    print(f"  Flights: {len(normal_delays):,}")
    print(f"  Mean delay: {normal_delays.mean():.2f} min")
    print(f"  Median delay: {normal_delays.median():.2f} min")
    print(f"  Std dev: {normal_delays.std():.2f} min")
    print(f"  Delay rate: {(normal_delays >= 15).sum() / len(normal_delays) * 100:.1f}%")
    
    print(f"\nHoliday Periods:")
    print(f"  Flights: {len(holiday_delays):,}")
    print(f"  Mean delay: {holiday_delays.mean():.2f} min")
    print(f"  Median delay: {holiday_delays.median():.2f} min")
    print(f"  Std dev: {holiday_delays.std():.2f} min")
    print(f"  Delay rate: {(holiday_delays >= 15).sum() / len(holiday_delays) * 100:.1f}%")
    
    # Statistical tests
    from scipy.stats import ttest_ind, ks_2samp
    
    t_stat, p_value_t = ttest_ind(normal_delays, holiday_delays)
    ks_stat, p_value_ks = ks_2samp(normal_delays, holiday_delays)
    
    print(f"\nStatistical Tests:")
    print(f"  T-test: t={t_stat:.3f}, p={p_value_t:.6f}")
    print(f"  KS-test: statistic={ks_stat:.3f}, p={p_value_ks:.6f}")
    print(f"  Significant difference: {'YES' if p_value_t < 0.05 else 'NO'} (α=0.05)")
else:
    print("⚠️  Cannot perform holiday comparison without date column")

In [ ]:
# Create distribution comparison
if date_col:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Overlapping histograms
    axes[0, 0].hist(normal_delays_plot, bins=100, alpha=0.5, label='Normal Days', 
                    color='blue', density=True)
    axes[0, 0].hist(holiday_delays_plot, bins=100, alpha=0.5, label='Holiday Periods', 
                    color='red', density=True)
    axes[0, 0].axvline(normal_delays.mean(), color='blue', linestyle='--', 
                       linewidth=2, label=f'Normal Mean ({normal_delays.mean():.1f})')
    axes[0, 0].axvline(holiday_delays.mean(), color='red', linestyle='--', 
                       linewidth=2, label=f'Holiday Mean ({holiday_delays.mean():.1f})')
    axes[0, 0].set_xlabel('Departure Delay (minutes)', fontsize=11)
    axes[0, 0].set_ylabel('Density', fontsize=11)
    axes[0, 0].set_title('Distribution Comparison: Holiday vs Normal Days', 
                         fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # Plot 2: KDE comparison
    normal_delays_plot.plot.kde(ax=axes[0, 1], color='blue', linewidth=2, label='Normal Days')
    holiday_delays_plot.plot.kde(ax=axes[0, 1], color='red', linewidth=2, label='Holiday Periods')
    axes[0, 1].axvline(15, color='orange', linestyle='--', linewidth=2, label='Delay Threshold')
    axes[0, 1].set_xlabel('Departure Delay (minutes)', fontsize=11)
    axes[0, 1].set_ylabel('Density', fontsize=11)
    axes[0, 1].set_title('Kernel Density Estimate: Smoothed Distribution', 
                         fontsize=12, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    
    # Plot 3: Box plot comparison
    data_to_plot = [normal_delays_plot, holiday_delays_plot]
    bp = axes[1, 0].boxplot(data_to_plot, labels=['Normal Days', 'Holiday Periods'],
                            patch_artist=True, showmeans=True)
    bp['boxes'][0].set_facecolor('lightblue')
    bp['boxes'][1].set_facecolor('lightcoral')
    axes[1, 0].set_ylabel('Departure Delay (minutes)', fontsize=11)
    axes[1, 0].set_title('Box Plot Comparison: Quartiles and Outliers', 
                         fontsize=12, fontweight='bold')
    axes[1, 0].grid(axis='y', alpha=0.3)
    axes[1, 0].axhline(15, color='orange', linestyle='--', linewidth=2, alpha=0.7)
    
    # Plot 4: Percentile comparison
    percentiles = np.arange(0, 101, 5)
    normal_percentiles = np.percentile(normal_delays, percentiles)
    holiday_percentiles = np.percentile(holiday_delays, percentiles)
    
    axes[1, 1].plot(percentiles, normal_percentiles, 'o-', color='blue', 
                    linewidth=2, label='Normal Days', markersize=4)
    axes[1, 1].plot(percentiles, holiday_percentiles, 's-', color='red', 
                    linewidth=2, label='Holiday Periods', markersize=4)
    axes[1, 1].axhline(15, color='orange', linestyle='--', linewidth=2, label='Delay Threshold')
    axes[1, 1].set_xlabel('Percentile', fontsize=11)
    axes[1, 1].set_ylabel('Delay (minutes)', fontsize=11)
    axes[1, 1].set_title('Percentile Comparison: Distribution Shape', 
                         fontsize=12, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)
    
    plt.suptitle('Comprehensive Statistical Comparison: Holiday Disruption Impact\n' +
                 f'p-value={p_value_t:.6f} (Statistically Significant Difference)',
                 fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    save_viz(fig, 'anomaly_02_distribution_comparison')
    plt.show()
else:
    print("⚠️  Skipping distribution comparison")

**Interpretation**:

*Statistical comparison reveals:*
- Shift in central tendency (mean/median)
- Change in distribution shape (wider/narrower)
- Tail behavior (more extreme delays during holidays?)
- Statistical significance of the difference

## Visualization 2C: Individual Holiday Deep Dive

**Purpose**: Compare impact of different holidays on delay patterns.

**Complexity**: 
- Multi-panel comparison across holiday types
- Before/during/after analysis for each holiday
- Magnitude of disruption quantification

In [ ]:
# Analyze each holiday individually
if date_col:
    holiday_analysis = []
    
    for holiday_name, dates in HOLIDAYS_2024.items():
        holiday_dates_pd = [pd.to_datetime(d) for d in dates]
        
        # Get data for this holiday
        holiday_data = df[df[date_col].isin(holiday_dates_pd)]
        
        if len(holiday_data) > 0:
            # Calculate before period (7 days before)
            before_start = holiday_dates_pd[0] - timedelta(days=7)
            before_end = holiday_dates_pd[0] - timedelta(days=1)
            before_data = df[df[date_col].between(before_start, before_end)]
            
            # Calculate after period (7 days after)
            after_start = holiday_dates_pd[-1] + timedelta(days=1)
            after_end = holiday_dates_pd[-1] + timedelta(days=7)
            after_data = df[df[date_col].between(after_start, after_end)]
            
            holiday_analysis.append({
                'Holiday': holiday_name,
                'Before_Mean': before_data['DEP_DELAY'].mean() if len(before_data) > 0 else np.nan,
                'During_Mean': holiday_data['DEP_DELAY'].mean(),
                'After_Mean': after_data['DEP_DELAY'].mean() if len(after_data) > 0 else np.nan,
                'Before_Rate': (before_data['DEP_DELAY'] >= 15).sum() / len(before_data) * 100 if len(before_data) > 0 else np.nan,
                'During_Rate': (holiday_data['DEP_DELAY'] >= 15).sum() / len(holiday_data) * 100,
                'After_Rate': (after_data['DEP_DELAY'] >= 15).sum() / len(after_data) * 100 if len(after_data) > 0 else np.nan,
                'Flights_During': len(holiday_data)
            })
    
    holiday_df = pd.DataFrame(holiday_analysis)
    holiday_df['Disruption_Magnitude'] = holiday_df['During_Mean'] - holiday_df['Before_Mean']
    holiday_df = holiday_df.sort_values('Disruption_Magnitude', ascending=False)
    
    print("\n=== INDIVIDUAL HOLIDAY ANALYSIS ===")
    print(holiday_df[['Holiday', 'Before_Mean', 'During_Mean', 'After_Mean', 'Disruption_Magnitude']].to_string(index=False))
else:
    print("⚠️  Cannot perform individual holiday analysis")

In [ ]:
# Create before/during/after comparison visualization
if date_col and len(holiday_analysis) > 0:
    fig = go.Figure()
    
    x_labels = holiday_df['Holiday'].tolist()
    
    # Before bars
    fig.add_trace(go.Bar(
        name='Week Before',
        x=x_labels,
        y=holiday_df['Before_Mean'],
        marker_color='lightblue',
        text=holiday_df['Before_Mean'].round(1),
        textposition='outside'
    ))
    
    # During bars
    fig.add_trace(go.Bar(
        name='During Holiday',
        x=x_labels,
        y=holiday_df['During_Mean'],
        marker_color='red',
        text=holiday_df['During_Mean'].round(1),
        textposition='outside'
    ))
    
    # After bars
    fig.add_trace(go.Bar(
        name='Week After',
        x=x_labels,
        y=holiday_df['After_Mean'],
        marker_color='lightgreen',
        text=holiday_df['After_Mean'].round(1),
        textposition='outside'
    ))
    
    fig.update_layout(
        title='<b>Holiday Disruption Patterns: Before/During/After Comparison</b><br>' +
              '<sub>Mean Departure Delay (minutes) | Sorted by Disruption Magnitude</sub>',
        xaxis_title='Holiday Period',
        yaxis_title='Mean Departure Delay (minutes)',
        barmode='group',
        height=600,
        width=1400,
        showlegend=True
    )
    
    # Add baseline line
    baseline = normal_delays.mean() if date_col else 0
    fig.add_hline(y=baseline, line_dash='dash', line_color='gray', 
                  annotation_text=f'Normal Baseline ({baseline:.1f} min)',
                  annotation_position='right')
    
    save_viz(fig, 'anomaly_03_holiday_comparison')
    fig.show()
else:
    print("⚠️  Skipping holiday comparison")

**Interpretation**:

*Holiday-specific analysis reveals:*
- Which holidays cause the most severe disruptions
- Recovery patterns (how quickly delays normalize)
- Pre-holiday build-up vs post-holiday recovery
- Relative impact of different holiday types

## Visualization 2D: Delay Pattern Distortion Matrix

**Purpose**: Show how holidays distort the normal time-of-day and day-of-week patterns.

**Complexity**: 
- Dual heatmaps (normal vs holiday)
- Hour × Day of Week matrix
- Difference/distortion visualization

In [ ]:
# Create hour × day of week analysis
if date_col and 'DEP_TIME' in df.columns:
    # Extract hour and day of week
    df['Hour'] = df['DEP_TIME'].apply(
        lambda x: int(x) // 100 if pd.notna(x) and isinstance(x, (int, float)) else np.nan
    )
    df['DayOfWeek'] = df[date_col].dt.dayofweek  # 0=Monday, 6=Sunday
    df['DayName'] = df[date_col].dt.day_name()
    
    # Create matrices for normal vs holiday
    normal_matrix = df[~df['Is_Holiday']].pivot_table(
        values='DEP_DELAY',
        index='Hour',
        columns='DayName',
        aggfunc='mean'
    )
    
    holiday_matrix = df[df['Is_Holiday']].pivot_table(
        values='DEP_DELAY',
        index='Hour',
        columns='DayName',
        aggfunc='mean'
    )
    
    # Reorder columns by day of week
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    normal_matrix = normal_matrix.reindex(columns=day_order)
    holiday_matrix = holiday_matrix.reindex(columns=day_order)
    
    # Calculate difference
    distortion_matrix = holiday_matrix - normal_matrix
    
    # Create triple visualization
    fig, axes = plt.subplots(1, 3, figsize=(20, 8))
    
    # Normal pattern
    sns.heatmap(normal_matrix, annot=False, fmt='.1f', cmap='RdYlGn_r', 
                center=0, ax=axes[0], cbar_kws={'label': 'Mean Delay (min)'},
                vmin=-10, vmax=30)
    axes[0].set_title('Normal Days: Delay Pattern\n(Hour × Day of Week)', 
                      fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Day of Week', fontsize=11)
    axes[0].set_ylabel('Hour of Day', fontsize=11)
    
    # Holiday pattern
    sns.heatmap(holiday_matrix, annot=False, fmt='.1f', cmap='RdYlGn_r', 
                center=0, ax=axes[1], cbar_kws={'label': 'Mean Delay (min)'},
                vmin=-10, vmax=30)
    axes[1].set_title('Holiday Periods: Delay Pattern\n(Hour × Day of Week)', 
                      fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Day of Week', fontsize=11)
    axes[1].set_ylabel('Hour of Day', fontsize=11)
    
    # Distortion (difference)
    sns.heatmap(distortion_matrix, annot=False, fmt='.1f', cmap='RdBu_r', 
                center=0, ax=axes[2], cbar_kws={'label': 'Delay Increase (min)'},
                vmin=-15, vmax=15)
    axes[2].set_title('Holiday Distortion\n(Holiday - Normal)', 
                      fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Day of Week', fontsize=11)
    axes[2].set_ylabel('Hour of Day', fontsize=11)
    
    plt.suptitle('How Holidays Distort Normal Delay Patterns: Temporal Heat Analysis',
                 fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    save_viz(fig, 'anomaly_04_pattern_distortion')
    plt.show()
else:
    print("⚠️  Cannot create pattern distortion analysis")

**Interpretation**:

*Pattern distortion reveals:*
- Which hours/days are most affected by holidays
- Whether holidays amplify existing peak-hour problems
- Temporal shifts in delay patterns during holidays
- Systematic changes in operational rhythms

---

# SUMMARY OF KEY FINDINGS

## Geospatial Bottleneck Insights

*[To be completed after running with actual data]*

**Major Hub Performance:**
- Best performing hub: [AIRPORT]
- Worst performing hub: [AIRPORT]
- Hub vs non-hub difference: [X]% higher delays

**Critical Routes:**
- Most delayed route: [ORIGIN]-[DEST] ([X] min avg delay)
- Geographic clusters: [REGION] shows concentration of delays
- Network centrality: [AIRPORT] appears in most bottleneck routes

## Holiday Disruption Insights

*[To be completed after running with actual data]*

**Most Disruptive Holidays:**
1. [HOLIDAY]: +[X]% delay increase
2. [HOLIDAY]: +[X]% delay increase
3. [HOLIDAY]: +[X]% delay increase

**Pattern Changes:**
- Normal baseline: [X] min avg delay, [Y]% delay rate
- Holiday average: [X] min avg delay, [Y]% delay rate
- Statistical significance: p < [VALUE]

**Recovery Patterns:**
- Typical recovery time: [X] days post-holiday
- Pre-holiday buildup: [X] days before peak

---

**✓ Advanced Geospatial & Anomaly Analysis Complete**